In [2]:
import sys
sys.path.insert(0,'/hdd/miles/velocity_cfm/dnnlib')
from util import Vanderpol,DoubleCircles,DoubleSDE,Lorenz63,Lorenz96,Rossler,projection,Balls

import os
import shutil
from datetime import datetime
from pathlib import Path

from lfads_torch.run_model import run_model

import torch
import numpy as np
import matplotlib.pyplot as plt

In [6]:
gen = Lorenz63()
dataTrain = gen.generate(n=500,T=1,dt=0.01,sigma=0.5)
dataTest = gen.generate(n=100,T=1,dt=0.01,sigma=0.5)
p = projection(origDim=3,newDim=5,projType='double swish',temp=1.5)

projectedTrain,projectedTest = [p.project(traj) for traj in dataTrain], [p.project(traj) for traj in dataTest]
dataset = {'train_data': np.stack(projectedTrain,axis=0),'valid_data': np.stack(projectedTest,axis=0),'data_dim':5,'num_steps':len(projectedTrain[0])}

In [7]:
import h5py
def save_lfads_data(train,test,gt_train,gt_test,filename):
    with h5py.File(filename,'w') as f:
        f.create_dataset('train_encod_data',data=train)
        f.create_dataset('train_recon_data',data=train)
        f.create_dataset('valid_encod_data',data=test)
        f.create_dataset('valid_recon_data',data=test)
        f.create_dataset('ground_truth_train',data=gt_train)
        f.create_dataset('ground_truth_valid',data=gt_test)
        f.create_dataset('encode_dim',data=gt_train.shape[-1])
        f.create_dataset('data_dim',data=train.shape[-1])

    

    

    

In [9]:
save_lfads_data(np.stack(projectedTrain,axis=0),np.stack(projectedTest,axis=0),\
                np.stack(dataTrain,axis=0),np.stack(dataTest,axis=0),\
                filename='/home/miles/isilon/All_Staff/miles/comparison_models/lfads/lorenz63_5d.h5')

In [19]:
project_str='lfads-test-lorenz'
dataset_str='lorenz63_5d'
run_tag= datetime.now().strftime("%y%m%d") + "example_time"
run_dir = Path("/home/miles/isilon/All_Staff/miles/comparison_models/lfads/runs") / project_str / dataset_str / run_tag
overwrite=True

In [21]:
os.getcwd()

'/home/miles/isilon/All_Staff/miles/comparison_models/lfads/runs/lfads-test-lorenz/lorenz63_5d/240815example_time'

In [26]:
# Overwrite the directory if necessary
if run_dir.exists() and overwrite:
    shutil.rmtree(run_dir)
run_dir.mkdir(parents=True)
# Copy this script into the run directory
#shutil.copyfile(__file__, run_dir / Path(__file__).name)
# Switch to the `RUN_DIR` and train the model
os.chdir(run_dir)
print(os.getcwd())
print(os.path.isfile('../configs/lfads_circles_model.yaml'))
run_model(
    overrides={
        "datamodule": dataset_str,
        "model": dataset_str,
    },
    config_path="../configs/lfads_circles_model.yaml",
)

/home/miles/isilon/All_Staff/miles/comparison_models/lfads/runs/lfads-test-lorenz/lorenz63_5d/240815example_time
True


MissingConfigException: Cannot find primary config 'lfads_circles_model.yaml'. Check that it's in your config search path.

Config search path:
	provider=hydra, path=pkg://hydra.conf
	provider=main, path=file:///hdd/miles/lfads-torch/configs
	provider=schema, path=structured://

In [15]:
datasets = {'lorenz63': dataset}

In [16]:
from run_lfads import train, flags, hps_dict_to_obj

In [17]:
def make_lfads_dict(flags_dict,true_dim,data_dim,dist='gaussian'):
    # output parameters
    flags_dict['output_dist'] = dist
    # generation parameters
    flags_dict['ic_dim'] = true_dim #initial conditions dimension
    flags_dict['factors_dim'] = true_dim # output of generator, priod to decoding
    flags_dict['ic_enc_dim'] = 2*true_dim #hidden dimension of encoding RNN

    flags_dict['gen_dim'] = 4*true_dim # hidden dimension of generator RNN

    flags_dict['ci_enc_dim'] = 4*true_dim # hidden size for encoder of control inputs
    flags_dict['con_dim'] =  2*true_dim # hidden size of controller
    
    flags_dict['batch_size']= 128

    return hps_dict_to_obj(flags_dict)

    
    
    
    

In [20]:
d = flags.FLAGS.flag_values_dict()
hps = make_lfads_dict(d,true_dim=3,data_dim=5)
hps.kind = 'train'
hps.dataset_names = []
hps.dataset_dims = {}
for key in datasets:
    hps.dataset_names.append(key)
    hps.dataset_dims[key] = datasets[key]['data_dim']
hps.num_steps = list(datasets.values())[0]['num_steps']
hps.ndatasets = len(hps.dataset_names)
if hps.num_steps_for_gen_ic > hps.num_steps:
    hps.num_steps_for_gen_ic = hps.num_steps

In [21]:
import tensorflow as tf
config = tf.compat.v1.ConfigProto(allow_soft_placement=True,
                          log_device_placement=False)
config.gpu_options.allow_growth=True


In [22]:
sess = tf.compat.v1.Session(config=config)
with sess.as_default():
    with tf.device(hps.device):
        train(hps,dataset)

Building graph...


2024-08-13 16:59:02.972039: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-08-13 16:59:02.972299: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-08-13 16:59:02.972480: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

AttributeError: module 'tensorflow' has no attribute 'placeholder'